In [ ]:
!pip install -q -U langchain langchain-community langchain-google-genai langchain-text-splitters chromadb pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("Key loaded")

Key loaded


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")
print(llm.invoke("Explain RAG in one line").text)

RAG (Retrieval-Augmented Generation) is a technique that supercharges AI by searching a trusted external database for relevant facts *before* generating an answer, ensuring its responses are accurate, up-to-date, and grounded in real data.


In [ ]:
from google.colab import files
upload=files.upload()
pdf_name=list(upload.keys())[0]
print(pdf_name)

Saving css_leave_rules_1972_30pages.pdf to css_leave_rules_1972_30pages.pdf
css_leave_rules_1972_30pages.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(pdf_name)
pages = loader.load()
print(len(pages))

/tmp/ipykernel_6699/762220028.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


30


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitters=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitters.split_documents(pages)
print(len(chunks))

91


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

test = embeddings.embed_query("casual leave rules")
print(len(test))
print(test[:5])

3072
[0.0052514863, 0.01588698, -0.0035696812, -0.050522283, 0.004791874]


In [ ]:
from langchain_community.vectorstores import Chroma
import time

vectorstores = Chroma(embedding_function=embeddings, collection_name="leave_rule")
batch_size = 20

for i in range(0, len(chunks), batch_size):
    vectorstores.add_documents(chunks[i : i + batch_size])
    print(f"saved {min(i + batch_size, len(chunks))}/{len(chunks)}")
    time.sleep(2)

/tmp/ipykernel_6699/545794934.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstores = Chroma(embedding_function=embeddings, collection_name="leave_rule")


saved 20/91
saved 40/91
saved 60/91
saved 80/91
saved 91/91


In [ ]:
vectorstore=vectorstores
print(vectorstore._collection.count())

91


In [ ]:
questions="how many days of casual leaves are allowed in year "
result=vectorstore.similarity_search(questions,k=3)
for r in result:
  print("page",r.metadata["page"]+1)
  print(r.page_content[:300])
  print("-----")

page 8
provisions of Rule 31. 
 
11. Combination of different kinds of leave 
 
Except as otherwise provided in these rules, any kind of leave under these 
rules may be granted in combination with or in continuation of any other  
kind of leave. 
 
EXPLANATION.- Casual leave which is not recognized as leav
-----
page 19
-:: 17 ::- 
 
 
Provided that where the earned leave at the credit of Government 
servant as on the last day of December of June is 300 days or less but 
more than 285 days, the advance credit of 15 days earned leave on first 
day of January or July to be afforded in the manner indicated under sub -
-----
page 22
Department) shall be credited with half pay leave in advance, in two 
instalments of ten days each on the first day of January and July of every  
calendar year]. 
 
(2)(a) The leave shall be credited to the said leave a ccount at the rate of 5/3  
days for each completed calendar month of service w
-----


In [ ]:
def ask(question):
    results = vectorstore.similarity_search(question, k=4)

    context = ""
    pages = []
    for r in results:
        context = context + r.page_content + "\n\n"
        pages.append(r.metadata["page"] + 1)

    prompt = "Answer ONLY from this context. If not found, say 'I could not find this in the document.'\n\n"
    prompt = prompt + context + "Question: " + question

    answer = llm.invoke(prompt).text
    print("Sources: pages", sorted(set(pages)))
    return answer

In [ ]:
ask("how many days of leave are allowed in a year ")
ask("How is earned leave credited to a government servant?")
ask("How many days of half pay leave are credited in advance in each instalment?")
ask("What is the capital of France?")

Sources: pages [19, 20, 21]
Sources: pages [18, 20, 21]
Sources: pages [22]
Sources: pages [1, 3, 4]


'I could not find this in the document.'

In [ ]:
!pip install -q gradio

In [ ]:
def ask(question):
    results = vectorstore.similarity_search(question, k=4)

    context = ""
    pages = []
    for r in results:
        context = context + r.page_content + "\n\n"
        pages.append(r.metadata["page"] + 1)

    prompt = "Answer ONLY from this context. If not found, say 'I could not find this in the document.'\n\n"
    prompt = prompt + context + "Question: " + question

    answer=llm.invoke(prompt).text
    return answer+"\n\n sorce :pages"+str(sorted(set(pages)))

In [ ]:
import gradio as gr
demo=gr.Interface(
    fn=ask,
    inputs=gr.Textbox(label="ask the question about leave rules"),
    outputs=gr.Textbox(label="answer"),
    title="leave rule Q & A bot",
    description="ask the question about leave rules",
)
demo.launch(share=True,debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a27a6a395477bfbbd0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a27a6a395477bfbbd0.gradio.live
